In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.optimize import curve_fit
import json, os

df = pd.read_csv("data/global_ads_performance_dataset.csv")
df['date'] = pd.to_datetime(df['date'])

daily = df.groupby(['date','platform']).agg(
    ad_spend=('ad_spend','sum'),
    revenue=('revenue','sum'),
    conversions=('conversions','sum')
).reset_index()

print("=== AGGREGATED DAILY DATA (per platform) ===")
print(daily.groupby('platform').size())
print(daily.groupby('platform')[['ad_spend','revenue']].describe().round(2))

print("\n=== LOG-LOG ELASTICITY REGRESSION per platform ===")
elasticity_results = {}
for p in daily['platform'].unique():
    sub = daily[daily['platform']==p].copy()
    X = sm.add_constant(np.log(sub['ad_spend']))
    y = np.log(sub['revenue'])
    model = sm.OLS(y, X).fit()
    beta = model.params.iloc[1]
    se = model.bse.iloc[1]
    ci_low, ci_high = model.conf_int().iloc[1]
    r2 = model.rsquared
    print(f"\n{p}:")
    print(f"  Elasticity (beta) = {beta:.3f}  [95% CI: {ci_low:.3f} - {ci_high:.3f}]")
    print(f"  R-squared = {r2:.3f}")
    print(f"  Interpretation: mỗi 1% tăng spend => {beta:.2f}% tăng revenue")
    elasticity_results[p] = {
        "beta": round(beta,3), "ci_low": round(ci_low,3), "ci_high": round(ci_high,3),
        "r_squared": round(r2,3), "intercept": round(model.params.iloc[0],3)
    }


def mm_curve(spend, vmax, k):
    return vmax * spend / (k + spend)

print("\n=== SATURATION CURVE FIT (Michaelis-Menten) ===")
curve_params = {}
curve_points = {}
for p in daily['platform'].unique():
    sub = daily[daily['platform']==p].sort_values('ad_spend')
    x, y = sub['ad_spend'].values, sub['revenue'].values
    try:
        popt, _ = curve_fit(mm_curve, x, y, p0=[y.max()*2, x.mean()], maxfev=10000)
        vmax, k = popt
        print(f"{p}: Vmax={vmax:.0f}, K={k:.0f}")
        curve_params[p] = {"vmax": round(vmax,2), "k": round(k,2)}
        
        x_smooth = np.linspace(x.min(), x.max()*1.3, 50)
        y_smooth = mm_curve(x_smooth, vmax, k)
        curve_points[p] = {"spend": x_smooth.round(2).tolist(), "revenue": y_smooth.round(2).tolist()}
    except RuntimeError:
        print(f"{p}: Curve fit failed, dùng log-log elasticity thay thế")


print("\n=== MARGINAL ROAS tại mức chi tiêu trung bình hiện tại ===")
marginal_results = {}
for p in daily['platform'].unique():
    sub = daily[daily['platform']==p]
    avg_spend = sub['ad_spend'].mean()
    avg_revenue = sub['revenue'].mean()
    blended_roas = avg_revenue / avg_spend
    beta = elasticity_results[p]['beta']
    marginal_roas = beta * blended_roas
    print(f"{p}: avg_daily_spend=${avg_spend:.0f}, blended_ROAS={blended_roas:.2f}, marginal_ROAS={marginal_roas:.2f}")
    marginal_results[p] = {
        "avg_daily_spend": round(avg_spend,2),
        "blended_roas": round(blended_roas,2),
        "marginal_roas": round(marginal_roas,2)
    }

os.makedirs("dashboard_data", exist_ok=True)
output = {
    "elasticity": elasticity_results,
    "saturation_curve_params": curve_params,
    "saturation_curve_points": curve_points,
    "marginal_roas": marginal_results
}
with open("dashboard_data/step3_spend_response.json", "w") as f:
    json.dump(output, f, indent=2)
print("\n✅ Saved: dashboard_data/step3_spend_response.json")

=== AGGREGATED DAILY DATA (per platform) ===
platform
Google Ads    307
Meta Ads      308
TikTok Ads    258
dtype: int64
           ad_spend                                                           \
              count      mean       std     min      25%       50%       75%   
platform                                                                       
Google Ads    307.0  20681.66  15773.96  336.30  8834.33  17260.20  29754.18   
Meta Ads      308.0   6837.86   5534.10  138.60  2622.88   5453.08   9712.12   
TikTok Ads    258.0  10284.57   8327.81  329.28  4329.92   7819.64  14583.12   

                     revenue                                                  \
                 max   count      mean       std     min       25%       50%   
platform                                                                       
Google Ads  89869.03   307.0  71771.16  64008.62  161.02  24917.26  51455.77   
Meta Ads    34527.23   308.0  38720.93  34324.30  490.60  11104.92  29791.78  